# IE643 - Connector pilot: cross-attention vs prefix (CE only)

**Version `pilot-v1`.** Trains the student twice under identical conditions, changing only the connector mode, and compares them on the **val** split (test is never touched).

| | |
|---|---|
| Encoder | `vit_tiny_patch16_384.augreg_in21k_ft_in1k` @ 384 (same as the baselines notebook) |
| Decoder | DistilGPT2 |
| Modes | `xattn` (MLP connector -> cross-attention in every block) vs `prefix` (MLP connector -> 32-query resampler -> prefix tokens) |
| Loss | cross-entropy on the ground-truth impression only (no KD / feature / clinical terms, so the teacher cache is not needed) |
| Schedule | epoch 1 = align (pretrained encoder + decoder frozen), epochs 2-5 = full fine-tune; AdamW, warm-up + cosine, fp16 |
| Metrics | val CE every epoch; after training, greedy decoding of all val images -> BLEU / ROUGE-L / METEOR / CIDEr + CheXbert F1 on the generated text |

**Decision rule (fixed before running):** primary metric = CheXbert 14-label micro-F1 of generated vs ground-truth impression (both labelled by CheXbert, uncertain = negative). Choose `prefix` only if the paired-bootstrap 95% CI of (prefix - xattn) is entirely above 0; otherwise keep `xattn`.

**How to run**
1. `MODE = "smoke"`, Accelerator **None**, Internet **On** -> Save Version -> Save & Run All (Commit). Free (CPU), ~15-25 min. Tiny subset, real pretrained weights; checks downloads, the decoder, CheXbert loading and every metric. Last line must be `SMOKE OK`.
2. `MODE = "pilot"`, Accelerator **GPU T4 x2**, Internet **On** -> Save & Run All (Commit). Both modes train at the same time, one per GPU. The log prints `TIMING` lines after 30 steps with the real s/step and the projected hours.

**Inputs:** notebook outputs **IE643 Cleaning Step 7** (`reports_clean.csv`) and **IE643 Cleaning** (`images_512.npy`).

In [ ]:
MODE = "smoke"                 # "smoke" (CPU, free) -> "pilot" (GPU T4 x2, committed)
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
CSV_PATH = None                # None = known Step 7 output path
NPY_PATH = None

IMG = 384
ENCODER = "vit_tiny_patch16_384.augreg_in21k_ft_in1k"
MODES = ["xattn", "prefix"]
EPOCHS = 5
ALIGN_EPOCHS = 1               # first epoch(s): pretrained encoder + decoder frozen
BATCH = 16
LR_NEW, LR_PRE, WD = 2e-4, 3e-5, 0.01
WARMUP, CLIP = 200, 1.0
MAX_LEN = None                 # None = 95th percentile of tokenised impression length + 1 (EOS), capped at 256
SEED = 0
SAVE_WEIGHTS = True            # best-epoch weights (fp16, ~200 MB per mode) for reuse
BUDGET_H = 10.5                # workers stop training at this wall time and still evaluate (12 h session limit)
PRETRAINED = True              # keep True on Kaggle
print("notebook version: pilot-v1 | MODE =", MODE)

In [ ]:
MODEL_PY = r'''"""IE643 student: ViT-Tiny encoder -> connector -> DistilGPT2 decoder (+ distillation / clinical heads).

Two ways to condition the decoder on the image (cfg.mode):
  "xattn"  : cross-attention inside every DistilGPT2 block (CvT2DistilGPT2 style). Main model.
  "prefix" : a Perceiver-style resampler turns the patch tokens into `n_prefix` vectors that are
             prepended to the text embeddings (ClipCap / LLaVA style). Ablation. With
             set_stage("align") kept for the whole run this is the frozen-LM prefix-tuning ablation.

Conventions
  * BOS == EOS == 50256 (GPT-2 has no BOS). Student vocab = 50257, teacher ids >= 50257 are dropped in the loss.
  * forward() returns logits ALIGNED with labels: logits[:, j] predicts labels[:, j]
    (input = [BOS, t0..t_{n-1}], labels = [t0..t_{n-1}, EOS]). No internal shift anywhere.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2Config, GPT2LMHeadModel

EOS_ID = 50256
VOCAB = 50257


@dataclass
class StudentConfig:
    # encoder
    # Main setting: native-384 checkpoint (576 patch tokens, 24x24 grid -> clean 4x4 pooling).
    # 224 ablation: encoder_name="vit_tiny_patch16_224.augreg_in21k_ft_in1k", img_size=224.
    # Keep encoder_name/img_size identical to the NN-retrieval baseline you compare against.
    encoder_name: str = "vit_tiny_patch16_384.augreg_in21k_ft_in1k"
    img_size: int = 384
    pretrained_enc: bool = True
    # decoder
    decoder_name: str = "distilgpt2"
    pretrained_dec: bool = True
    mode: str = "xattn"            # "xattn" | "prefix"
    n_prefix: int = 32             # prefix mode: number of visual prefix tokens
    resampler_heads: int = 8
    # distillation targets (cached teacher features)
    d_grid: int = 1024             # siglip_grid4  [16, 1024]
    d_mean: int = 1024             # siglip_mean   [1024]
    d_proj: int = 2560             # proj_mean     [2560]
    grid_hw: int = 4
    # clinical head
    n_clin: int = 14
    drop: float = 0.1
    max_len: int = 192             # max text tokens incl. BOS. EDA: median impression 86 tok, p95 187 (pre-cleaning);
                                   # set from the p95 of ntok_final in reports_clean.csv before training.


class Connector(nn.Module):
    """Linear(D_v->H) -> GELU -> Linear(H->H) -> LayerNorm (LLaVA-1.5 style 2-layer MLP)."""

    def __init__(self, d_in: int, d_out: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_out), nn.GELU(), nn.Linear(d_out, d_out))
        self.norm = nn.LayerNorm(d_out)

    def forward(self, x):
        return self.norm(self.net(x))


class Resampler(nn.Module):
    """m learned queries cross-attend to the visual tokens -> [B, m, H] (Perceiver / Q-Former lite)."""

    def __init__(self, d: int, n_queries: int, n_heads: int, drop: float = 0.0):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(n_queries, d) * 0.02)
        self.ln_q, self.ln_kv, self.ln_ff = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=drop, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, kv):
        q = self.queries.unsqueeze(0).expand(kv.size(0), -1, -1)
        kv = self.ln_kv(kv)
        x = q + self.attn(self.ln_q(q), kv, kv, need_weights=False)[0]
        return x + self.ff(self.ln_ff(x))


class XRayReportStudent(nn.Module):
    def __init__(self, cfg: StudentConfig):
        super().__init__()
        assert cfg.mode in ("xattn", "prefix")
        self.cfg = cfg

        # ---- encoder -------------------------------------------------------------------------
        self.encoder = timm.create_model(
            cfg.encoder_name, pretrained=cfg.pretrained_enc, num_classes=0, img_size=cfg.img_size
        )
        self.n_prefix_tok = getattr(self.encoder, "num_prefix_tokens", 1)  # CLS (+ reg tokens)
        d_v = self.encoder.num_features
        self.d_v = d_v
        self.enc_cfg = self.encoder.pretrained_cfg  # mean/std for preprocessing

        # ---- decoder -------------------------------------------------------------------------
        xattn = cfg.mode == "xattn"
        if cfg.pretrained_dec:
            self.decoder = GPT2LMHeadModel.from_pretrained(cfg.decoder_name, add_cross_attention=xattn)
        else:  # offline / unit-test path: distilgpt2 geometry, random init
            gcfg = GPT2Config(n_layer=6, n_head=12, n_embd=768, vocab_size=VOCAB, n_positions=1024,
                              add_cross_attention=xattn)
            self.decoder = GPT2LMHeadModel(gcfg)
        self.d_t = self.decoder.config.n_embd

        # ---- connector -----------------------------------------------------------------------
        self.connector = Connector(d_v, self.d_t)
        self.resampler = (
            Resampler(self.d_t, cfg.n_prefix, cfg.resampler_heads, cfg.drop) if cfg.mode == "prefix" else None
        )

        # ---- auxiliary heads (train only; dropped at export) --------------------------------
        self.grid_head = nn.Sequential(nn.LayerNorm(d_v), nn.Linear(d_v, cfg.d_grid))
        self.mean_head = nn.Sequential(nn.LayerNorm(d_v), nn.Linear(d_v, cfg.d_mean))
        self.proj_head = nn.Sequential(nn.LayerNorm(d_v), nn.Linear(d_v, cfg.d_proj))
        self.clin_head = nn.Sequential(
            nn.LayerNorm(2 * d_v), nn.Dropout(cfg.drop), nn.Linear(2 * d_v, d_v), nn.GELU(),
            nn.Linear(d_v, cfg.n_clin),
        )

    # ------------------------------------------------------------------------------------------
    def encode(self, images):
        """images [B,3,H,W] (already normalised) -> dict of visual tokens and head predictions."""
        feats = self.encoder.forward_features(images)              # [B, npt+N, D]
        cls = feats[:, 0]
        patches = feats[:, self.n_prefix_tok:]                     # [B, N, D]
        B, N, D = patches.shape
        g = int(math.isqrt(N))
        assert g * g == N, "patch tokens must form a square grid"
        mean = patches.mean(1)
        grid = F.adaptive_avg_pool2d(patches.transpose(1, 2).reshape(B, D, g, g), self.cfg.grid_hw)
        grid = grid.flatten(2).transpose(1, 2)                     # [B, 16, D] row-major
        memory = self.connector(feats)                             # [B, npt+N, H]
        if self.resampler is not None:
            memory = self.resampler(memory)                        # [B, m, H]
        return {"memory": memory, "cls": cls, "mean": mean, "grid": grid}

    def heads(self, enc):
        return {
            "grid_pred": self.grid_head(enc["grid"]),              # [B,16,d_grid]
            "mean_pred": self.mean_head(enc["mean"]),              # [B,d_mean]
            "proj_pred": self.proj_head(enc["mean"]),              # [B,d_proj]
            "clin_logits": self.clin_head(torch.cat([enc["cls"], enc["mean"]], -1)),  # [B,n_clin]
        }

    def forward(self, images, input_ids, attention_mask=None, with_heads=True):
        enc = self.encode(images)
        mem = enc["memory"]
        if self.cfg.mode == "xattn":
            out = self.decoder(input_ids=input_ids, attention_mask=attention_mask,
                               encoder_hidden_states=mem, use_cache=False)
            logits = out.logits
        else:
            m = mem.size(1)
            emb = torch.cat([mem, self.decoder.transformer.wte(input_ids)], 1)
            am = None
            if attention_mask is not None:
                am = torch.cat([attention_mask.new_ones(attention_mask.size(0), m), attention_mask], 1)
            out = self.decoder(inputs_embeds=emb, attention_mask=am, use_cache=False)
            logits = out.logits[:, m:]                              # drop prefix positions
        res = {"logits": logits}
        if with_heads:
            res.update(self.heads(enc))
        return res

    # ------------------------------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, images, max_new_tokens=60, num_beams=3, no_repeat_ngram_size=3,
                 length_penalty=1.0, **kw):
        """Beam (or greedy if num_beams=1) decoding with KV cache. Returns list[list[int]] without BOS/EOS."""
        was_training = self.training
        self.eval()
        enc = self.encode(images)
        mem = enc["memory"]
        B = mem.size(0)
        bos = torch.full((B, 1), EOS_ID, dtype=torch.long, device=mem.device)
        common = dict(max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False,
                      no_repeat_ngram_size=no_repeat_ngram_size, length_penalty=length_penalty,
                      eos_token_id=EOS_ID, pad_token_id=EOS_ID, early_stopping=True, **kw)
        if self.cfg.mode == "xattn":
            ids = self.decoder.generate(input_ids=bos, encoder_hidden_states=mem, **common)
            seqs = ids[:, 1:]                                       # strip BOS
        else:
            emb = torch.cat([mem, self.decoder.transformer.wte(bos)], 1)
            seqs = self.decoder.generate(inputs_embeds=emb, **common)  # returns only new tokens
        out = []
        for s in seqs.tolist():
            if EOS_ID in s:
                s = s[: s.index(EOS_ID)]
            out.append(s)
        self.train(was_training)
        return out

    @torch.no_grad()
    def greedy(self, images, max_new_tokens=128):
        """Greedy decoding with the decoder's KV cache, written by hand so it does not depend on the
        transformers version's generate() handling of encoder_hidden_states / inputs_embeds.
        Returns list[list[int]] without BOS/EOS."""
        was_training = self.training
        self.eval()
        mem = self.encode(images)["memory"]
        B = mem.size(0)
        cur = torch.full((B, 1), EOS_ID, dtype=torch.long, device=mem.device)
        if self.cfg.mode == "xattn":
            out = self.decoder(input_ids=cur, encoder_hidden_states=mem, use_cache=True)
        else:
            out = self.decoder(inputs_embeds=torch.cat([mem, self.decoder.transformer.wte(cur)], 1), use_cache=True)
        seqs = []
        done = torch.zeros(B, dtype=torch.bool, device=mem.device)
        for _ in range(max_new_tokens):
            nxt = out.logits[:, -1, :VOCAB].argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, EOS_ID), nxt)
            seqs.append(nxt)
            done |= nxt == EOS_ID
            if bool(done.all()):
                break
            kw = dict(input_ids=nxt[:, None], past_key_values=out.past_key_values, use_cache=True)
            if self.cfg.mode == "xattn":
                kw["encoder_hidden_states"] = mem
            out = self.decoder(**kw)
        res = []
        for s in torch.stack(seqs, 1).tolist():
            res.append(s[: s.index(EOS_ID)] if EOS_ID in s else s)
        self.train(was_training)
        return res

    @torch.no_grad()
    def greedy(self, images, max_new_tokens=128):
        """Greedy decoding with an explicit KV-cache loop (does not rely on HF generate(), whose handling of
        encoder_hidden_states for decoder-only GPT-2 differs across transformers versions).
        Returns (list[list[int]] without BOS/EOS, list[bool] hit_cap = no EOS within max_new_tokens)."""
        was_training = self.training
        self.eval()
        enc = self.encode(images)
        mem = enc["memory"]
        B, dev = mem.size(0), mem.device
        xattn = self.cfg.mode == "xattn"
        bos = torch.full((B, 1), EOS_ID, dtype=torch.long, device=dev)
        if xattn:
            out = self.decoder(input_ids=bos, encoder_hidden_states=mem, use_cache=True)
        else:
            out = self.decoder(inputs_embeds=torch.cat([mem, self.decoder.transformer.wte(bos)], 1), use_cache=True)
        past, nxt = out.past_key_values, out.logits[:, -1].argmax(-1)
        done = torch.zeros(B, dtype=torch.bool, device=dev)
        steps = []
        for step in range(max_new_tokens):
            nxt = torch.where(done, torch.full_like(nxt, EOS_ID), nxt)
            steps.append(nxt)
            done = done | (nxt == EOS_ID)
            if bool(done.all()) or step == max_new_tokens - 1:
                break
            kw = dict(input_ids=nxt[:, None], past_key_values=past, use_cache=True)
            if xattn:
                kw["encoder_hidden_states"] = mem
            out = self.decoder(**kw)
            past, nxt = out.past_key_values, out.logits[:, -1].argmax(-1)
        toks = torch.stack(steps, 1).tolist()
        seqs, capped = [], []
        for s in toks:
            if EOS_ID in s:
                seqs.append(s[: s.index(EOS_ID)]); capped.append(False)
            else:
                seqs.append(s); capped.append(True)
        self.train(was_training)
        return seqs, capped

    # ------------------------------------------------------------------------------------------
    def _is_pretrained(self, name: str) -> bool:
        if name.startswith("encoder."):
            return True
        return name.startswith("decoder.") and "crossattn" not in name and "ln_cross_attn" not in name

    def set_stage(self, stage: str):
        """'align': freeze pretrained encoder+decoder, train connector / cross-attn / resampler / heads.
        'full'  : train everything (use a lower LR on pretrained parts via param_groups)."""
        assert stage in ("align", "full")
        for n, p in self.named_parameters():
            p.requires_grad = True if stage == "full" else not self._is_pretrained(n)

    def param_groups(self, lr_new=1e-4, lr_pre=3e-5, wd=0.01):
        buckets = {(pre, dec): [] for pre in (True, False) for dec in (True, False)}
        for n, p in self.named_parameters():
            if p.requires_grad:
                buckets[(self._is_pretrained(n), p.ndim >= 2 and "wpe" not in n)].append(p)
        groups = []
        for (pre, dec), ps in buckets.items():
            if ps:
                groups.append({"params": ps, "lr": lr_pre if pre else lr_new, "weight_decay": wd if dec else 0.0})
        return groups

    def size_report(self):
        def cnt(m):
            return sum(p.numel() for p in m.parameters())
        wte = self.decoder.transformer.wte.weight.numel()
        r = {
            "encoder": cnt(self.encoder),
            "connector": cnt(self.connector) + (cnt(self.resampler) if self.resampler is not None else 0),
            "decoder_total": cnt(self.decoder),
            "decoder_token_embedding": wte,
            "train_only_heads": sum(cnt(m) for m in (self.grid_head, self.mean_head, self.proj_head, self.clin_head)),
        }
        r["deploy_total"] = r["encoder"] + r["connector"] + r["decoder_total"]
        r["deploy_total_after_vocab_prune_to_8k"] = r["deploy_total"] - wte + 8000 * self.d_t
        return r
'''

WORKER_PY = r'''"""IE643 pilot worker: trains ONE student connector mode with cross-entropy only, then greedy-decodes the val set.

Usage: python pilot_worker.py <cfg.json> <mode> <out_dir>
Run one process per GPU (CUDA_VISIBLE_DEVICES set by the notebook) or on CPU for the smoke test.
Writes to out_dir: worker.log, curve.jsonl, summary.json, val_predictions.csv, best.pt (fp16 state dict, optional).
"""
import json, math, os, random, sys, time

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

cfg_path, MODE, OUT = sys.argv[1], sys.argv[2], sys.argv[3]
C = json.load(open(cfg_path))
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from model import StudentConfig, XRayReportStudent, EOS_ID  # noqa: E402

LOG = open(f"{OUT}/worker.log", "a", buffering=1)
T0 = time.time()


def log(*a):
    s = f"[{MODE} {time.time() - T0:7.0f}s] " + " ".join(str(x) for x in a)
    print(s, flush=True)
    LOG.write(s + "\n")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


seed_all(C["seed"])
dev = torch.device("cuda" if torch.cuda.is_available() and not C["force_cpu"] else "cpu")
amp = dev.type == "cuda"
log(f"device {dev} | torch {torch.__version__} | cfg {json.dumps({k: v for k, v in C.items() if k != 'note'})}")

# ---------------------------------------------------------------- data ---------------------------------
D = torch.load(C["data_file"], weights_only=False)  # {train: {idx, ids}, val: {idx, ids, text}}
imgs = np.load(C["npy"], mmap_mode="r")
L = C["max_len"]


def encode_ids(ids):
    """input = [BOS] + t[:L-1]; labels = t[:L-1] + [EOS or the next real token if truncated]."""
    t = ids[: L - 1]
    nxt = ids[L - 1] if len(ids) > L - 1 else EOS_ID
    return [EOS_ID] + t, t + [nxt]


class DS(torch.utils.data.Dataset):
    def __init__(self, split):
        self.idx, self.ids = D[split]["idx"], D[split]["ids"]

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        inp, lab = encode_ids(self.ids[i])
        return torch.from_numpy(np.array(imgs[self.idx[i]])), inp, lab, i


def collate(b):
    n = max(len(x[1]) for x in b)
    inp = torch.full((len(b), n), EOS_ID, dtype=torch.long)
    lab = torch.full((len(b), n), -100, dtype=torch.long)
    att = torch.zeros((len(b), n), dtype=torch.long)
    for j, (_, a, l, _) in enumerate(b):
        inp[j, : len(a)] = torch.tensor(a); lab[j, : len(l)] = torch.tensor(l); att[j, : len(a)] = 1
    img = torch.stack([x[0] for x in b])
    return img, inp, att, lab, torch.tensor([x[3] for x in b])


def loader(split, shuffle, bs):
    g = torch.Generator(); g.manual_seed(C["seed"])
    return torch.utils.data.DataLoader(DS(split), batch_size=bs, shuffle=shuffle, collate_fn=collate, generator=g,
                                       num_workers=C["num_workers"], pin_memory=amp, drop_last=shuffle,
                                       persistent_workers=C["num_workers"] > 0)


# ---------------------------------------------------------------- model --------------------------------
cfg = StudentConfig(mode=MODE, encoder_name=C["encoder"], img_size=C["img"], max_len=L,
                    pretrained_enc=C["pretrained"], pretrained_dec=C["pretrained"])
model = XRayReportStudent(cfg).to(dev)
MEAN = torch.tensor(model.enc_cfg["mean"], device=dev).view(1, 3, 1, 1)
STD = torch.tensor(model.enc_cfg["std"], device=dev).view(1, 3, 1, 1)
log(f"encoder {C['encoder']} @ {C['img']} | normalisation mean {model.enc_cfg['mean']} std {model.enc_cfg['std']}")
log("size", {k: f"{v / 1e6:.2f}M" for k, v in model.size_report().items()})


def prep(img_u8, train):
    """uint8 [B,512,512] -> normalised [B,3,IMG,IMG] on device; light geometric aug for training (no flips:
    laterality matters in CXR reports)."""
    x = img_u8.to(dev, non_blocking=True).float().div_(255).unsqueeze(1)
    x = F.interpolate(x, size=(C["img"], C["img"]), mode="bilinear", antialias=True, align_corners=False)
    if train and C["aug"]:
        B = x.size(0)
        ang = (torch.rand(B, device=dev) * 2 - 1) * math.radians(C["aug_rot_deg"])
        sc = 1 + (torch.rand(B, device=dev) * 2 - 1) * C["aug_scale"]
        tx, ty = [(torch.rand(B, device=dev) * 2 - 1) * C["aug_shift"] for _ in range(2)]
        th = torch.stack([torch.stack([torch.cos(ang) / sc, -torch.sin(ang) / sc, tx], 1),
                          torch.stack([torch.sin(ang) / sc, torch.cos(ang) / sc, ty], 1)], 1)
        grid = F.affine_grid(th, x.shape, align_corners=False)
        x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    return (x.expand(-1, 3, -1, -1) - MEAN) / STD


def ce(logits, labels):
    return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100,
                           reduction="sum"), int((labels != -100).sum())


# ---------------------------------------------------------------- sanity checks (smoke) ----------------
if C["checks"]:
    model.eval()
    dl = loader("val", False, 2)
    img, inp, att, lab, _ = next(iter(dl))
    x = prep(img, False)
    with torch.no_grad():
        seqs, _ = model.greedy(x, max_new_tokens=6)
        for i in range(2):
            s = seqs[i]
            lg = model(x[i:i + 1], torch.tensor([[EOS_ID] + s], device=dev), with_heads=False)["logits"][0]
            assert lg.argmax(-1).tolist()[: len(s)] == s, "KV-cache greedy != no-cache recompute"
        l1 = model(x, inp.to(dev), att.to(dev), with_heads=False)["logits"]
        l2 = model(x.flip(0), inp.to(dev), att.to(dev), with_heads=False)["logits"]
        assert float((l1 - l2).abs().max()) > 1e-4, "decoder output does not depend on the image"
        inp2 = inp.clone(); inp2[att == 0] = 7
        l3 = model(x, inp2.to(dev), att.to(dev), with_heads=False)["logits"]
        m = att.bool().to(dev)
        assert float((l1 - l3)[m].abs().max()) < 1e-3, "padding leaks into valid positions"
    log("CHECKS OK: cached greedy == recompute, output depends on image, padding masked")

# ---------------------------------------------------------------- optimiser / schedule -----------------
model.set_stage("full")
opt = torch.optim.AdamW(model.param_groups(C["lr_new"], C["lr_pre"], C["wd"]), betas=(0.9, 0.999), eps=1e-8)
base_lrs = [g["lr"] for g in opt.param_groups]
tr_dl = loader("train", True, C["batch"])
va_dl = loader("val", False, C["eval_batch"])
steps_per_epoch = len(tr_dl) if C["max_steps_per_epoch"] <= 0 else min(len(tr_dl), C["max_steps_per_epoch"])
total = steps_per_epoch * C["epochs"]
warm = min(C["warmup"], max(1, total // 10))


def lr_factor(step):
    if step < warm:
        return (step + 1) / warm
    p = (step - warm) / max(1, total - warm)
    return 0.5 * (1 + math.cos(math.pi * min(1.0, p)))


scaler = torch.amp.GradScaler("cuda", enabled=amp)


@torch.no_grad()
def evaluate():
    model.eval()
    tot, n = 0.0, 0
    for img, inp, att, lab, _ in va_dl:
        with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):
            lg = model(prep(img, False), inp.to(dev), att.to(dev), with_heads=False)["logits"]
        s, k = ce(lg, lab.to(dev))
        tot += float(s); n += k
    model.train()
    return tot / max(n, 1)


deadline = T0 + C["budget_h"] * 3600
step, best, best_ep, stopped_early = 0, float("inf"), -1, False
curve = open(f"{OUT}/curve.jsonl", "w", buffering=1)
v0 = evaluate()
log(f"epoch 0 (before training) val_CE {v0:.4f} ppl {math.exp(min(v0, 20)):.1f}")
curve.write(json.dumps({"epoch": 0, "step": 0, "val_ce": v0, "train_ce": None}) + "\n")
model.train()
for ep in range(1, C["epochs"] + 1):
    stage = "align" if ep <= C["align_epochs"] else "full"
    model.set_stage(stage)
    ntr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"epoch {ep} stage={stage} trainable={ntr / 1e6:.2f}M steps={steps_per_epoch}")
    run_s, run_n, t_ep, t_win = 0.0, 0, time.time(), time.time()
    for it, (img, inp, att, lab, _) in enumerate(tr_dl):
        if it >= steps_per_epoch:
            break
        for g, b in zip(opt.param_groups, base_lrs):
            g["lr"] = b * lr_factor(step)
        with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):
            lg = model(prep(img, True), inp.to(dev), att.to(dev), with_heads=False)["logits"]
        s, k = ce(lg, lab.to(dev))
        loss = s / max(k, 1)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], C["clip"])
        scaler.step(opt); scaler.update()
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss at step {step}")
        run_s += float(s); run_n += k; step += 1
        if step == C["timing_steps"]:
            dt = (time.time() - t_ep) / C["timing_steps"]
            mem = torch.cuda.max_memory_allocated() / 2**30 if amp else 0
            log(f"TIMING {dt:.3f} s/step (batch {C['batch']}, first {C['timing_steps']} steps incl. warm-up) "
                f"-> ~{dt * steps_per_epoch / 60:.1f} min/epoch, ~{dt * total / 3600:.2f} h for {C['epochs']} epochs | "
                f"peak GPU mem {mem:.2f} GiB")
        if step % C["log_every"] == 0:
            log(f"  step {step}/{total} train_CE {run_s / max(run_n, 1):.4f} lr_new {opt.param_groups[0]['lr']:.2e} "
                f"grad_norm {float(gn):.2f} {(time.time() - t_win) / C['log_every']:.3f}s/step")
            t_win = time.time()
        if time.time() > deadline:
            stopped_early = True
            log("deadline reached: stopping training, going to evaluation")
            break
    v = evaluate()
    tr_ce = run_s / max(run_n, 1)
    log(f"epoch {ep} done in {(time.time() - t_ep) / 60:.1f} min | train_CE {tr_ce:.4f} | val_CE {v:.4f} "
        f"ppl {math.exp(min(v, 20)):.2f}")
    curve.write(json.dumps({"epoch": ep, "step": step, "stage": stage, "val_ce": v, "train_ce": tr_ce}) + "\n")
    if v < best:
        best, best_ep = v, ep
        best_state = {k: t.detach().to("cpu", copy=True) for k, t in model.state_dict().items()}
    if stopped_early:
        break

# ---------------------------------------------------------------- generation on val with best epoch ----
model.load_state_dict(best_state)
if C["save_weights"]:
    torch.save({k: (t.half() if t.is_floating_point() else t) for k, t in best_state.items()}, f"{OUT}/best.pt")
del best_state
model.eval()
from transformers import AutoTokenizer  # noqa: E402

tok = AutoTokenizer.from_pretrained("distilgpt2")
rows, t_gen = [], time.time()
with torch.no_grad():
    for img, inp, att, lab, ii in loader("val", False, C["gen_batch"]):
        with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):
            seqs, capped = model.greedy(prep(img, False), max_new_tokens=C["gen_max"])
        for j, s, cp in zip(ii.tolist(), seqs, capped):
            rows.append({"idx": int(D["val"]["idx"][j]), "gt": D["val"]["text"][j],
                         "pred": tok.decode(s, skip_special_tokens=True).strip(), "n_tok": len(s), "hit_cap": cp})
pred = pd.DataFrame(rows)
pred.to_csv(f"{OUT}/val_predictions.csv", index=False)
log(f"generated {len(pred)} val impressions in {(time.time() - t_gen) / 60:.1f} min | mean tokens "
    f"{pred.n_tok.mean():.1f} | hit cap {pred.hit_cap.mean():.1%} | distinct {pred.pred.nunique()}")
for r in pred.head(3).itertuples():
    log(f"  GT  : {r.gt[:160]!r}\n  PRED: {r.pred[:160]!r}")
summary = {"mode": MODE, "best_epoch": best_ep, "best_val_ce": best, "steps": step, "stopped_early": stopped_early,
           "params": model.size_report(), "train_minutes": None, "minutes_total": (time.time() - T0) / 60,
           "gen_mean_tokens": float(pred.n_tok.mean()), "gen_hit_cap_frac": float(pred.hit_cap.mean()),
           "gen_distinct": int(pred.pred.nunique()), "cfg": C}
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1)
log("DONE")
'''

CHEXBERT_PY = r'''"""CheXbert labeler (Smit et al. 2020) for scoring generated report text.

Same architecture and checkpoint as the `f1chexbert` package (StanfordAIMI/RRG_scorers/chexbert.pth) but:
  * strict weight loading (only the known `position_ids` buffer may differ between transformers versions),
  * batched inference,
  * returns the raw 4-way classes so both label conventions can be computed:
      head classes 0 blank, 1 positive, 2 negative, 3 uncertain  (No Finding head: 0 blank, 1 positive)
      "pos_only"  : positive = class 1; uncertain counts as negative  (matches the IE643 baselines)
      "rrg"       : positive = class 1 or 3 (uncertain counts as positive; f1chexbert / RRG-paper convention)
"""
import numpy as np
import torch
import torch.nn as nn

CHEXBERT_ORDER = ["Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema",
                  "Consolidation", "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other",
                  "Fracture", "Support Devices", "No Finding"]


class _Labeler(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        h = bert.config.hidden_size
        self.linear_heads = nn.ModuleList([nn.Linear(h, 4) for _ in range(13)] + [nn.Linear(h, 2)])

    def forward(self, ids, att):
        cls = self.bert(ids, attention_mask=att)[0][:, 0]
        return [hd(cls) for hd in self.linear_heads]


class CheXbert:
    def __init__(self, device="cpu", checkpoint=None, random_init=False):
        from transformers import AutoConfig, AutoModel, BertTokenizer
        self.device = torch.device(device)
        self.tok = BertTokenizer.from_pretrained("bert-base-uncased")
        self.model = _Labeler(AutoModel.from_config(AutoConfig.from_pretrained("bert-base-uncased")))
        if not random_init:
            if checkpoint is None:
                from huggingface_hub import hf_hub_download
                checkpoint = hf_hub_download("StanfordAIMI/RRG_scorers", "chexbert.pth")
            sd = torch.load(checkpoint, map_location="cpu", weights_only=False)
            sd = sd.get("model_state_dict", sd)
            sd = {k.replace("module.", "", 1): v for k, v in sd.items()}
            res = self.model.load_state_dict(sd, strict=False)
            bad_missing = [k for k in res.missing_keys if not k.endswith("position_ids")]
            bad_unexp = [k for k in res.unexpected_keys if not k.endswith("position_ids")]
            assert not bad_missing and not bad_unexp, f"CheXbert weights mismatch: missing {bad_missing[:5]} unexpected {bad_unexp[:5]}"
        self.model.to(self.device).eval()

    @staticmethod
    def _clean(t):
        return " ".join(str(t).strip().replace("\n", " ").split())

    @torch.no_grad()
    def classes(self, texts, bs=64):
        """-> int array [N,14] of raw classes in CHEXBERT_ORDER."""
        out = []
        for s in range(0, len(texts), bs):
            chunk = [self._clean(t) for t in texts[s:s + bs]]
            # [CLS] + wordpieces + [SEP], truncated to 512 (same ids as the reference tokenize -> encode_plus path;
            # encode_plus was removed in transformers 5)
            enc = [self.tok(t, add_special_tokens=True, truncation=True, max_length=512)["input_ids"] if t else
                   [self.tok.cls_token_id, self.tok.sep_token_id] for t in chunk]
            n = max(len(e) for e in enc)
            ids = torch.zeros(len(enc), n, dtype=torch.long)
            att = torch.zeros(len(enc), n, dtype=torch.float)
            for i, e in enumerate(enc):
                ids[i, :len(e)] = torch.tensor(e); att[i, :len(e)] = 1
            logits = self.model(ids.to(self.device), att.to(self.device))
            out.append(torch.stack([l.argmax(1) for l in logits], 1).cpu())
        return torch.cat(out).numpy()

    @staticmethod
    def binary(cls, labels, convention="pos_only"):
        """raw classes [N,14] -> 0/1 matrix [N,len(labels)] in the order of `labels`."""
        J = [CHEXBERT_ORDER.index(c) for c in labels]
        c = cls[:, J]
        pos = (c == 1) if convention == "pos_only" else ((c == 1) | (c == 3))
        return pos.astype(np.int8)
'''
print('modules embedded:', len(MODEL_PY), len(WORKER_PY), len(CHEXBERT_PY))

In [ ]:
import os, sys, json, glob, re, subprocess, time, shutil
import numpy as np, pandas as pd

try:
    import pycocoevalcap
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocoevalcap"], check=True)
import torch, timm, transformers
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | timm {timm.__version__} | transformers {transformers.__version__} "
      f"| java: {shutil.which('java') or 'NOT FOUND (METEOR skipped)'}")

SRC_DIR = f"{WORK}/pilot_src"
os.makedirs(SRC_DIR, exist_ok=True)
MODULES = {"model.py": MODEL_PY, "pilot_worker.py": WORKER_PY, "chexbert_labeler.py": CHEXBERT_PY}
for n, s in MODULES.items():
    open(f"{SRC_DIR}/{n}", "w").write(s)
sys.path.insert(0, SRC_DIR)

KNOWN_CSV = f"{INPUT_ROOT}/notebooks/jithendraampatt/ie643-cleaning-step-7/reports_clean.csv"
KNOWN_NPY = f"{INPUT_ROOT}/notebooks/jithendraampatt/ie643-cleaning/images_512.npy"

def resolve_data():
    csv, npy = CSV_PATH, NPY_PATH
    if csv is None and os.path.exists(KNOWN_CSV):
        csv = KNOWN_CSV
    if npy is None and os.path.exists(KNOWN_NPY):
        npy = KNOWN_NPY
    if csv is None:
        c = [p for p in glob.glob(f"{INPUT_ROOT}/**/reports_clean.csv", recursive=True)
             if {"impression_final", "use_for_training"} <= set(pd.read_csv(p, nrows=0).columns)]
        assert len(c) == 1, f"Need exactly one reports_clean.csv containing impression_final; found {c}. Attach 'IE643 Cleaning Step 7'."
        csv = c[0]
    if npy is None:
        c = glob.glob(f"{INPUT_ROOT}/**/images_512.npy", recursive=True)
        assert len(c) == 1, f"Need exactly one images_512.npy; found {c}. Attach 'IE643 Cleaning'."
        npy = c[0]
    assert os.path.exists(csv) and os.path.exists(npy), f"missing {csv} or {npy}"
    return csv, npy

CSV, NPY = resolve_data()
df = pd.read_csv(CSV, low_memory=False)
imgs = np.load(NPY, mmap_mode="r")
assert len(df) == imgs.shape[0] and (df["idx"].values == np.arange(len(df))).all(), "CSV and images_512.npy are not aligned"
df["impression_final"] = df["impression_final"].fillna("").astype(str)
df["use_for_training"] = df["use_for_training"].astype(str).str.lower().isin(["true", "1"])
U = df[df.use_for_training]
print(f"csv: {CSV}\nnpy: {NPY}\nrows {len(df)} | usable per split {U.subset.value_counts().to_dict()}")

LABELS = ["No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
          "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices"]
FIVE = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
assert all(c in df.columns for c in LABELS), [c for c in LABELS if c not in df.columns]

In [ ]:
# ---- tokenise impressions (DistilGPT2 = teacher token IDs), choose MAX_LEN, write the shared data file ----
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("distilgpt2")
tr = U[U.subset == "train"].reset_index(drop=True)
va = U[U.subset == "val"].reset_index(drop=True)
if MODE == "smoke":
    tr, va = tr.sample(48, random_state=SEED).reset_index(drop=True), va.sample(16, random_state=SEED).reset_index(drop=True)
ids_tr = [tok(t)["input_ids"] for t in tr.impression_final]
ids_va = [tok(t)["input_ids"] for t in va.impression_final]
lens = np.array([len(x) + 1 for x in ids_tr])                       # +1 for EOS
p50, p95, p99 = np.percentile(lens, [50, 95, 99])
if MAX_LEN is None:
    MAX_LEN = int(min(256, np.ceil(p95)))
print(f"impression tokens incl. EOS (train): median {p50:.0f}, p95 {p95:.0f}, p99 {p99:.0f}, max {lens.max()} "
      f"-> MAX_LEN = {MAX_LEN}; truncated: {(lens > MAX_LEN).mean():.1%} of train impressions")
DATA_FILE = f"{WORK}/pilot_data.pt"
torch.save({"train": {"idx": tr.idx.tolist(), "ids": ids_tr},
            "val": {"idx": va.idx.tolist(), "ids": ids_va, "text": va.impression_final.tolist()},
            "tokenisation": "distilgpt2, plain text (no leading space), no lower-casing"}, DATA_FILE)
print(f"train {len(tr)} | val {len(va)} -> {DATA_FILE}")

In [ ]:
# ---- launch one worker per mode (parallel on 2 GPUs in pilot mode; sequential on CPU in smoke mode) ----
smoke = MODE == "smoke"
CFG = {"seed": SEED, "npy": NPY, "data_file": DATA_FILE, "max_len": MAX_LEN, "encoder": ENCODER, "img": IMG,
       "pretrained": PRETRAINED, "force_cpu": smoke, "checks": smoke,
       "epochs": 2 if smoke else EPOCHS, "align_epochs": 1 if smoke else ALIGN_EPOCHS,
       "max_steps_per_epoch": 3 if smoke else 0, "batch": 4 if smoke else BATCH, "eval_batch": 8 if smoke else 32,
       "gen_batch": 8 if smoke else 64, "gen_max": 24 if smoke else MAX_LEN,
       "lr_new": LR_NEW, "lr_pre": LR_PRE, "wd": WD, "warmup": WARMUP, "clip": CLIP,
       "aug": True, "aug_rot_deg": 5.0, "aug_scale": 0.05, "aug_shift": 0.03,
       "num_workers": 2, "log_every": 2 if smoke else 100, "timing_steps": 3 if smoke else 30,
       "budget_h": BUDGET_H, "save_weights": SAVE_WEIGHTS and not smoke}
RUN_DIR = f"{WORK}/pilot_runs"
os.makedirs(RUN_DIR, exist_ok=True)
json.dump(CFG, open(f"{RUN_DIR}/cfg.json", "w"), indent=1)

def n_gpus():
    try:
        return len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
    except FileNotFoundError:
        return 0

NG = n_gpus()
print(f"GPUs visible: {NG}")
if not smoke:
    assert NG >= 1, "pilot mode needs a GPU accelerator (T4 x2). Use MODE='smoke' on CPU."

def launch(mode, gpu):
    out = f"{RUN_DIR}/{mode}"
    os.makedirs(out, exist_ok=True)
    env = dict(os.environ, TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")
    env["CUDA_VISIBLE_DEVICES"] = "" if gpu is None else str(gpu)
    f = open(f"{out}/stdout.txt", "w")
    p = subprocess.Popen([sys.executable, f"{SRC_DIR}/pilot_worker.py", f"{RUN_DIR}/cfg.json", mode, out],
                         stdout=f, stderr=subprocess.STDOUT, env=env)
    return p, f"{out}/stdout.txt"

def tail(path, n=1):
    try:
        return open(path).read().strip().splitlines()[-n:]
    except Exception:
        return []

def wait(procs, every):
    seen = {m: 0 for m in procs}
    while True:
        alive = False
        for m, (p, log) in procs.items():
            lines = open(log).read().splitlines() if os.path.exists(log) else []
            for ln in lines[seen[m]:]:
                if any(k in ln for k in ("TIMING", "epoch", "CHECKS", "generated", "DONE", "deadline", "Error", "error", "Traceback")):
                    print(ln, flush=True)
            seen[m] = len(lines)
            alive |= p.poll() is None
        if not alive:
            break
        time.sleep(every)
    for m, (p, log) in procs.items():
        if p.returncode != 0:
            print(f"\n----- {m} FAILED (exit {p.returncode}); last 40 log lines -----")
            print("\n".join(tail(log, 40)))
    assert all(p.returncode == 0 for p, _ in procs.values()), "a worker failed - see the log above"

t0 = time.time()
if smoke:
    for m in MODES:
        wait({m: launch(m, None)}, 5)
elif NG >= len(MODES):
    wait({m: launch(m, g) for g, m in enumerate(MODES)}, 60)
else:
    print("fewer GPUs than modes: running the modes one after the other on GPU 0")
    for m in MODES:
        wait({m: launch(m, 0)}, 60)
print(f"\nall workers finished in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# ---- metrics (same text normalisation + NLG code as the baselines notebook) ----
def text_tokens(t):
    return " ".join(re.findall(r"[a-z0-9]+", t.lower()))

def nlg_metrics(preds, refs):
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    g = {i: [text_tokens(r) or "empty"] for i, r in enumerate(refs)}
    p = {i: [text_tokens(x) or "empty"] for i, x in enumerate(preds)}
    out = {}
    try:
        b, _ = Bleu(4).compute_score(g, p, verbose=0)
    except TypeError:
        b, _ = Bleu(4).compute_score(g, p)
    out.update({f"BLEU-{k + 1}": float(v) for k, v in enumerate(b)})
    out["ROUGE-L"] = float(Rouge().compute_score(g, p)[0])
    out["CIDEr"] = float(Cider().compute_score(g, p)[0])
    try:
        from pycocoevalcap.meteor.meteor import Meteor
        out["METEOR"] = float(Meteor().compute_score(g, p)[0])
    except Exception as e:
        out["METEOR"] = None
        print("METEOR skipped (needs Java):", repr(e)[:200])
    return out

def label_matrix(rows):
    m = np.zeros((len(rows), len(LABELS)), dtype=np.int8)
    for j, c in enumerate(LABELS):
        if c == "No Finding" and "y_no_finding" in rows.columns:
            m[:, j] = (rows["y_no_finding"].values == 1)
        else:
            m[:, j] = (rows[c].values == 1)
    return m

def micro_f1(P, G, cols=LABELS):
    J = [LABELS.index(c) for c in cols]
    P, G = P[:, J].astype(bool), G[:, J].astype(bool)
    tp = (P & G).sum(); fp = (P & ~G).sum(); fn = (~P & G).sum()
    return float(2 * tp / max(2 * tp + fp + fn, 1))

def boot_ci(P, G, cols, n=1000, seed=0):
    rs = np.random.RandomState(seed)
    v = [micro_f1(P[s], G[s], cols) for s in (rs.randint(0, len(P), len(P)) for _ in range(n))]
    return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]

def paired_boot(Pa, Pb, G, cols, n=2000, seed=0):
    """CI of F1(b) - F1(a) resampling the SAME studies for both models."""
    rs = np.random.RandomState(seed); d = []
    for _ in range(n):
        s = rs.randint(0, len(G), len(G))
        d.append(micro_f1(Pb[s], G[s], cols) - micro_f1(Pa[s], G[s], cols))
    return float(np.mean(d)), [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))]
print("metric smoke test:", {k: (round(v, 3) if v is not None else None) for k, v in nlg_metrics(["no acute disease ."], ["no acute disease ."]).items()})

In [ ]:
# ---- evaluation + decision ----
from chexbert_labeler import CheXbert, CHEXBERT_ORDER
dev = "cuda" if torch.cuda.is_available() else "cpu"
cxb = CheXbert(device=dev, random_init=not PRETRAINED)
P = {m: pd.read_csv(f"{RUN_DIR}/{m}/val_predictions.csv", keep_default_na=False) for m in MODES}
order = P[MODES[0]].idx.tolist()
assert all(P[m].idx.tolist() == order for m in MODES), "val prediction rows differ between modes"
vrows = df.set_index("idx").loc[order].reset_index()
G_csv = label_matrix(vrows)
g_cls = cxb.classes(vrows.impression_final.tolist())
G_cxb = CheXbert.binary(g_cls, LABELS, "pos_only")
G_rrg = CheXbert.binary(g_cls, LABELS, "rrg")
agree = micro_f1(G_cxb, G_csv)
print(f"sanity: CheXbert(GT impression) vs CSV labels micro-F1 = {agree:.3f} "
      "(CSV labels come from the full report; expect clearly above chance, roughly 0.5-0.9)")
if PRETRAINED and len(order) >= 100:
    assert agree > 0.3, "CheXbert labels barely agree with the CSV labels - labeler weights probably not loaded correctly"

R, PM = {}, {}
for m in MODES:
    pred = P[m].pred.astype(str).tolist()
    cls = cxb.classes(pred)
    PM[m] = {"pos": CheXbert.binary(cls, LABELS, "pos_only"), "rrg": CheXbert.binary(cls, LABELS, "rrg")}
    curve = [json.loads(l) for l in open(f"{RUN_DIR}/{m}/curve.jsonl")]
    summ = json.load(open(f"{RUN_DIR}/{m}/summary.json"))
    r = nlg_metrics(pred, vrows.impression_final.tolist())
    r.update({"CE14_F1": micro_f1(PM[m]["pos"], G_cxb), "CE5_F1": micro_f1(PM[m]["pos"], G_cxb, FIVE),
              "CE14_F1_95CI": boot_ci(PM[m]["pos"], G_cxb, LABELS),
              "CE14_F1_vs_CSV": micro_f1(PM[m]["pos"], G_csv), "CE5_F1_vs_CSV": micro_f1(PM[m]["pos"], G_csv, FIVE),
              "CE14_F1_rrg": micro_f1(PM[m]["rrg"], G_rrg), "CE5_F1_rrg": micro_f1(PM[m]["rrg"], G_rrg, FIVE),
              "best_val_CE": summ["best_val_ce"], "best_epoch": summ["best_epoch"],
              "val_CE_curve": [round(c["val_ce"], 4) for c in curve],
              "params_deploy_M": round(summ["params"]["deploy_total"] / 1e6, 2),
              "gen_mean_tokens": summ["gen_mean_tokens"], "gen_hit_cap": summ["gen_hit_cap_frac"],
              "distinct_preds": summ["gen_distinct"], "worker_minutes": round(summ["minutes_total"], 1)})
    R[m] = r

show = ["BLEU-1", "BLEU-4", "ROUGE-L", "METEOR", "CIDEr", "CE14_F1", "CE5_F1", "CE14_F1_vs_CSV", "CE5_F1_vs_CSV",
        "CE14_F1_rrg", "best_val_CE", "best_epoch", "gen_mean_tokens", "gen_hit_cap", "distinct_preds", "params_deploy_M", "worker_minutes"]
print("\n===== pilot results on val (%d studies) =====" % len(order))
print(pd.DataFrame({m: {k: R[m][k] for k in show} for m in MODES}).round(4).to_string())
print("\nval CE per epoch (epoch 0 = before training):")
for m in MODES:
    print(f"  {m:7s}", R[m]["val_CE_curve"])
print("CE14_F1 95% CI:", {m: [round(x, 3) for x in R[m]["CE14_F1_95CI"]] for m in MODES})

a, b = MODES[0], MODES[1]                              # xattn, prefix
diff, ci = paired_boot(PM[a]["pos"], PM[b]["pos"], G_cxb, LABELS)
still_falling = {m: (len(R[m]["val_CE_curve"]) > 2 and R[m]["val_CE_curve"][-2] - R[m]["val_CE_curve"][-1] > 0.01 * R[m]["val_CE_curve"][-2]) for m in MODES}
choice = b if ci[0] > 0 else a
print(f"\npaired bootstrap CE14_F1({b}) - CE14_F1({a}) = {diff:+.4f}, 95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]")
print("val CE still falling >1% in the last epoch:", still_falling)
print(f"DECISION (rule fixed in advance): use '{choice}'" + ("" if choice == b else
      f" ({b} is not better beyond noise)") + (" | NOTE: curves still falling, pilot may be too short to separate them"
      if any(still_falling.values()) else ""))
json.dump({"version": "pilot-v1", "mode": MODE, "results": R, "paired_diff_CE14": {"mean": diff, "ci95": ci},
           "decision": choice, "chexbert_vs_csv_agreement": agree, "cfg": CFG, "n_val": len(order)},
          open(f"{WORK}/pilot_results.json", "w"), indent=1)
print("\nSMOKE OK - switch MODE to 'pilot' and commit on GPU T4 x2" if MODE == "smoke" else "\nPILOT DONE -> pilot_results.json")